# Bài 20: Truy cập và Lọc dữ liệu trên Google Earth Engine

Google Earth Engine (GEE) là một trong những nền tảng điện toán đám mây mạnh mẽ nhất thế giới dành cho phân tích dữ liệu địa không gian, cung cấp quyền truy cập vào hàng nghìn bộ dữ liệu vệ tinh, khí hậu và địa lý, miễn phí cho nghiên cứu và giáo dục theo một cách thống nhất. Trước khi học bày này, bạn cần phải đăng kí tài khoản GEE nếu chưa có. Bạn có thể tham khảo hướng dẫn đăng kí theo video [link](https://www.youtube.com/watch?v=O9iyjs4w-8I) này.

## 20.1. Mục tiêu học tập

Sau khi hoàn thành bài học này, bạn sẽ có thể:

- Xác thực và khởi tạo GEE Python API
- Hiểu 5 đối tượng cơ bản của GEE: `ee.Image`, `ee.ImageCollection`, `ee.Feature`, `ee.FeatureCollection`
- Lọc `ImageCollection` theo thời gian, không gian và thuộc tính metadata
- Kết hợp nhiều bộ lọc để lấy đúng dữ liệu cần thiết
- Tra cứu và khám phá các tập dữ liệu phổ biến trong GEE Data Catalog

In [1]:
import ee # chưa có thì cài đặt bằng pip install earthengine-api
import geemap # chưa có thì cài đặt bằng pip install geemap
import geopandas as gpd # chưa có thì cài đặt bằng pip install geopandas
from shapely.geometry import mapping
ee.Authenticate()
ee.Initialize()

## 20.2. Các đối tượng (Objects) chính trong GEE

Khi làm việc với GEE, các đối tượng trong GEE đều được xử lý theo cơ chế server-side. Vì vậy, chúng ta cần sử dụng các hàm và phương thức có sẵn của GEE như map(), filter(), reduce(), thay vì thao tác trực tiếp bằng Python như với dữ liệu local (client-side). Chỉ khi dùng getInfo() thì dữ liệu mới được tải từ server về client.

Tổng quan dữ liệu trên GEE có thể tham khảo tại [đây](https://developers.google.com/earth-engine/datasets). Nếu bạn chưa có tài khoản, bạn có thể đăng kí theo hướng dẫn tại video [link](https://www.youtube.com/watch?v=O9iyjs4w-8I) này.

### 20.2.1. Tổng quan 4 đối tượng cơ bản

| Đối tượng | Mô tả | Ví dụ thực tế |
|---|---|---|
| `ee.Image` | Ảnh raster đơn lẻ (nhiều band) | DEM, một bức ảnh Sentinel-2 |
| `ee.ImageCollection` | Tập hợp nhiều ảnh (time-series) | Toàn bộ archive Landsat |
| `ee.Feature` | Đối tượng vector = geometry + properties | Ranh giới một tỉnh |
| `ee.FeatureCollection` | Tập hợp nhiều Feature | Ranh giới 63 tỉnh VN |

### 20.2.2. Truy cập dữ liệu 

- **`ee.Image`**
 
Ảnh raster đơn lẻ gồm một hoặc nhiều **band** (kênh phổ). Mỗi pixel lưu một giá trị số như DEM hoặc một ảnh Sentinel-2.

In [6]:
# ee.Image: SRTM DEM (độ cao địa hình toàn cầu, 30m)
srtm = ee.Image('USGS/SRTMGL1_003')
print(srtm.bandNames().getInfo()) # hiển thị tên các band dữ liệu trong ảnh
print(f"Các thuộc tính của ảnh: {srtm.propertyNames().getInfo()[:5]}") # hiển thị tên các thuộc tính của ảnh

['elevation']
Các thuộc tính của ảnh: ['system:visualization_0_min', 'type_name', 'keywords', 'thumb', 'description']


- **`ee.ImageCollection`** 
  
Tập hợp nhiều `ee.Image`, thường là **time-series** (cùng sensor, nhiều thời điểm). Đây là loại đối tượng bạn sẽ làm việc nhiều nhất trong GEE.

In [12]:
# Truy cập ee.ImageCollection: Sentinel-2 Surface Reflectance
s2col = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
# Lấy ảnh Sentinel-2 đầu tiên trong bộ sưu tập
s2img = s2col.first()
print(s2img.bandNames().getInfo()[:5]) # hiển thị tên các band dữ liệu trong ảnh
print(f"Các thuộc tính của ảnh: {s2img.propertyNames().getInfo()[:4]}") # hiển thị tên các thuộc tính của ảnh
print(f"Ngày chụp ảnh: {s2img.date().format('YYYY-MM-dd').getInfo()}") # hiển thị ngày chụp ảnh
print(f"Tên vệ tinh: {s2img.get('SPACECRAFT_NAME').getInfo()}") # hiển thị tên vệ tinh

['B1', 'B2', 'B3', 'B4', 'B5']
Các thuộc tính của ảnh: ['system:version', 'system:id', 'SPACECRAFT_NAME', 'SATURATED_DEFECTIVE_PIXEL_PERCENTAGE']
Ngày chụp ảnh: 2015-07-04
Tên vệ tinh: Sentinel-2A


- **`ee.Feature`** 
 
Một đối tượng địa lý vector kết hợp **geometry** + **properties** (thuộc tính dạng dict). Tương đương một hàng trong GeoDataFrame.

In [15]:
# ee.Feature: Tạo điểm đại diện cho Hà Nội
hanoi = ee.Feature(
    ee.Geometry.Point([105.8542, 21.0285]),
    {
        'name':       'Hà Nội',
        'population': 8246600,
        'is_capital': True,
    }
)
hanoi.getInfo()
print(f"Dân số Hà Nội: {hanoi.get('population').getInfo()} người") # hiển thị dân số Hà Nội

Dân số Hà Nội: 8246600 người


- **`ee.FeatureCollection`** 
 
`FeatureCollection` được tạo ra từ tập hợp nhiều `ee.Feature`. GEE cung cấp sẵn nhiều FeatureCollection công khai như ranh giới hành chính (GAUL, GADM), đường bờ biển, điểm quan trắc. Ngoài những dữ liệu có sẵn trên GEE, ta có thể chuyển một `shapefile` thành `FeatureCollection` như sau:

In [23]:
# Tạo feature collection từ dữ liệu geopandas dataframe 
germany = gpd.read_file('https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_DEU_1.json')

ee_features = []

for _, row in germany.iterrows():
    geom = row.geometry
    if geom is None:
        continue
    # Convert shapely geometry to GeoJSON mapping
    geojson = mapping(geom)
    ee_geom = ee.Geometry(geojson)

    # Convert row properties to a dictionary (excluding geometry)
    properties = row.drop(labels="geometry").to_dict()

    # Create ee.Feature
    ee_feature = ee.Feature(ee_geom, properties)
    ee_features.append(ee_feature)
fcol = ee.FeatureCollection(ee_features)
print(f"Số lượng feature trong collection: {fcol.size().getInfo()}")

Số lượng feature trong collection: 16


## 20.3. Lọc dữ liệu với Filters

GEE cung cấp các phương pháp lọc `ImageCollection` để lấy đúng dữ liệu cần thiết. Ba phương thức cốt lõi:

| Phương thức | Mục đích |
|---|---|
| `.filterDate(start, end)` | Lọc theo khoảng thời gian |
| `.filterBounds(geometry)` | Lọc ảnh giao với vùng AOI |
| `.filter(ee.Filter.xxx)` | Lọc theo thuộc tính metadata |

Các phương thức này có thể **chain** (nối tiếp nhau) và được thực thi lazy trên server GEE. Ví dụ dưới đây sử dụng bộ dữ liệu Sentinel-2 và sẽ thực hành các phép lọc trong GEE. Bạn có thể đọc thêm về ảnh Sentinel-2 trên [website](https://developers.google.com/earth-engine/datasets/catalog/sentinel-2).

### 20.3.1. Lọc theo vị trí 

Trong ví dụ này, chúng ta sử dụng dữ liệu Sentinel-2 để minh họa cách lọc dữ liệu theo vị trí. Các bộ dữ liệu khác cũng có thể được thực hiện tương tự.

In [2]:
# AOI: Hà Nội
aoi = ee.Geometry.BBox(105.7, 20.9, 106.0, 21.2)
# Đọc dữ liệu Sentinel-2 SR từ GEE và lọc theo lấy ảnh khu vực AOI (Hà Nội)
sen2col = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(aoi)
print(sen2col.size().getInfo()) # hiển thị số lượng ảnh Sentinel-2 SR có trong AOI Hà Nội

1134


### 20.3.2. Lọc theo ngày tháng

Để lọc ảnh theo thời gian, chúng ta sử dụng cú pháp `.filterDate(start, end)` nhằm chọn các ảnh nằm trong khoảng thời gian mong muốn. Thời gian được định dạng theo chuẩn `YYYY-MM-DD`.

In [29]:
# Ví dụ ta lọc dữ liệu Sentinel-2 trong năm 2026
sen2col = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
sen2_2026 = sen2col.filterDate('2026-01-01', '2026-12-31')
print(f"Số lượng ảnh Sentinel-2 SR trong năm 2026: {sen2_2026.size().getInfo()}")

Số lượng ảnh Sentinel-2 SR trong năm 2026: 1991274


### 20.3.3. Lọc theo thuộc tính

Để lọc ảnh theo thuộc tính, chung ta thường sử dụng cú pháp `.filter(ee.Filter.xxx())` để lọc dựa trên metadata của ảnh. Các filter phổ biến: `lt` (less than), `gt`, `eq`, `lte`, `gte`, `inList`, `stringContains`.

- **Lọc ảnh Sentinel-2 có độ mây bao phủ dưới 5%**

In [30]:
sen2col = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
# Đầu tiên phải xem thuộc tính của Collection.
print(f"Các thuộc tính của Sentinel-2 SR: {sen2col.first().propertyNames().getInfo()[:10]}") # hiển thị 10 thuộc tính đầu tiên của ảnh Sentinel-2 SR

Các thuộc tính của Sentinel-2 SR: ['system:version', 'system:id', 'SPACECRAFT_NAME', 'SATURATED_DEFECTIVE_PIXEL_PERCENTAGE', 'BOA_ADD_OFFSET_B12', 'CLOUD_SHADOW_PERCENTAGE', 'system:footprint', 'SENSOR_QUALITY', 'GENERATION_TIME', 'CLOUDY_PIXEL_OVER_LAND_PERCENTAGE']


In [3]:
# Cách 1 dùng ee.Filter.xx để lọc ảnh có giá trị thuộc tính CLOUDY_PIXEL_PERCENTAGE < 5
sen2col_cloudless = sen2col.filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 5))
print(f"Số lượng ảnh Sentinel-2 SR có độ che phủ mây < 5%: {sen2col_cloudless.size().getInfo()}")
# Cách 2 dùng 
sen2col_cloudless = sen2col.filterMetadata('CLOUDY_PIXEL_PERCENTAGE', 'less_than', 5)
print(f"Số lượng ảnh Sentinel-2 SR có độ che phủ mây < 5%: {sen2col_cloudless.size().getInfo()}")

Số lượng ảnh Sentinel-2 SR có độ che phủ mây < 5%: 77
Số lượng ảnh Sentinel-2 SR có độ che phủ mây < 5%: 77


### 20.3.4. Kết hợp nhiều Filters

Nên áp dụng các bộ lọc theo thứ tự date → bounds → metadata để Google Earth Engine tối ưu hiệu suất truy vấn. Cuối cùng, sử dụng .sort() để ưu tiên những ảnh có chất lượng tốt nhất hiển thị trước.

In [6]:
# Ví dụ lọc ảnh Sentinel-2 trong năm 2026 khu vực Hà Nội (AOI) và có độ che phủ mây < 5% và shadow < 5%
sen2col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
           .filterDate('2026-01-01', '2026-12-31')
           .filterBounds(aoi)
           .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 5))
           .filter(ee.Filter.lt('CLOUD_SHADOW_PERCENTAGE', 5))
           .sort('CLOUDY_PIXEL_PERCENTAGE', True) # sắp xếp theo độ che phủ mây tăng dần, ảnh ít mây nhất sẽ đứng đầu collection
           )
print(f"Số lượng ảnh Sentinel-2 SR trong năm 2026 khu vực Hà Nội có độ che phủ mây < 5% và shadow < 5%: {sen2col.size().getInfo()}")

Số lượng ảnh Sentinel-2 SR trong năm 2026 khu vực Hà Nội có độ che phủ mây < 5% và shadow < 5%: 4


## 20.4. Các bộ dữ liệu phổ biến trong GEE

GEE Data Catalog chứa hàng nghìn datasets. Dưới đây là các bộ dữ liệu thường dùng nhất trong nghiên cứu và ứng dụng địa không gian.

### 20.4.1. Dữ liệu Landsat

Landsat là chương trình vệ tinh quan sát Trái Đất lâu đời do NASA và USGS phối hợp vận hành, cung cấp ảnh quang học độ phân giải trung bình (30 m) với chuỗi dữ liệu liên tục từ năm 1972 đến nay. Trong Google Earth Engine, dữ liệu Landsat được sử dụng rộng rãi để nghiên cứu biến động lớp phủ đất, đô thị hóa, tài nguyên nước, nông nghiệp và môi trường theo thời gian.

In [ ]:
# Truy cập bộ sưu tập Landsat 9 Surface Reflectance trên GEE. Bộ sưu tập này có tên là "LANDSAT/LC09/C02/T1_L2" và chứa ảnh Landsat 9 đã được xử lý để có giá trị phản xạ bề mặt (Surface Reflectance). Ta có thể lọc bộ sưu tập này theo ngày, khu vực, hoặc các thuộc tính khác tương tự như cách đã làm với Sentinel-2.
landsat = ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")

- **Kiểm tra thông tin thuộc tính**

Mỗi ảnh sẽ có các thuộc tính khác nhau tùy thuộc vào loại ảnh và bộ sưu tập. Ví dụ, ảnh Landsat 9 SR có thể có các thuộc tính như CLOUDY_PIXEL_PERCENTAGE, trong khi ảnh Landsat 9 L2 có thể có các thuộc tính như RADIOMETRIC_QUALITY. Việc hiểu rõ các thuộc tính này sẽ giúp bạn lọc và chọn lựa ảnh phù hợp cho phân tích không gian của mình. Kiểm tra có thể được thực hiện như sau.

In [10]:
print(f"Các thuộc tính của ảnh Landsat 9 L2: {landsat.first().propertyNames().getInfo()[:10]}") # hiển thị 10 thuộc tính đầu tiên của ảnh Landsat 9 L2

Các thuộc tính của ảnh Landsat 9 L2: ['DATA_SOURCE_ELEVATION', 'WRS_TYPE', 'system:id', 'REFLECTANCE_ADD_BAND_1', 'REFLECTANCE_ADD_BAND_2', 'DATUM', 'REFLECTANCE_ADD_BAND_3', 'REFLECTANCE_ADD_BAND_4', 'REFLECTANCE_ADD_BAND_5', 'REFLECTANCE_ADD_BAND_6']


- **Kiểm tra các bands**

Mỗi ảnh có thể có một hoặc nhiều bands. Để kiểm tra số lượng bands trong một ảnh, bạn có thể sử dụng phương thức `bandNames()` của đối tượng ee.Image. Ví dụ, nếu bạn muốn kiểm tra số lượng bands trong ảnh Landsat-9, bạn có thể làm như sau:

In [11]:
print(f"Các bands của ảnh Landsat 9 L2: {landsat.first().bandNames().getInfo()[:10]}") # hiển thị 10 bands đầu tiên của ảnh Landsat 9 L2

Các bands của ảnh Landsat 9 L2: ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7', 'SR_QA_AEROSOL', 'ST_B10', 'ST_ATRAN']


### 20.4.2. Ảnh Sentinel-1 

Sentinel-1 là bộ dữ liệu ảnh radar khẩu độ tổng hợp (SAR) được cung cấp trong Google Earth Engine (GEE) từ vệ tinh Sentinel-1. Khác với ảnh quang học, Sentinel-1 sử dụng sóng vi ba nên có khả năng thu nhận dữ liệu cả ngày lẫn đêm và xuyên qua mây, giúp quan sát bề mặt Trái Đất liên tục trong mọi điều kiện thời tiết.

In [ ]:
# Để truy cập ảnh Sentinel-1, ta có thể sử dụng ee.ImageCollection("COPERNICUS/S1_GRD") để lấy toàn bộ bộ sưu tập Sentinel-1 GRD (Ground Range Detected). Sau đó, ta có thể lọc theo ngày, khu vực, hoặc các thuộc tính khác tương tự như cách đã làm với Sentinel-2 và Landsat.
sen1col = ee.ImageCollection("COPERNICUS/S1_GRD")

- **Kiểm tra thuộc tính của ảnh**
  
Mỗi ảnh sẽ có các thuộc tính khác nhau tùy thuộc vào loại ảnh và bộ sưu tập. Ví dụ như với Sentinel-1 như bên dưới.

In [ ]:
print(f"Các thuộc tính của ảnh Sentinel-1 GRD: {sen1col.first().propertyNames().getInfo()[:10]}") # hiển thị 10 thuộc tính đầu tiên của ảnh Sentinel-1 GRD

Các thuộc tính của ảnh Sentinel-1 GRD: ['system:version', 'system:id', 'SNAP_Graph_Processing_Framework_GPF_vers', 'SLC_Processing_facility_org', 'SLC_Processing_facility_country', 'GRD_Post_Processing_facility_org', 'transmitterReceiverPolarisation', 'GRD_Post_Processing_start', 'sliceNumber', 'GRD_Post_Processing_facility_name']


- **Kiểm tra bands của ảnh**

Mỗi ảnh có thể có một hoặc nhiều bands. Để kiểm tra số lượng bands trong một ảnh, bạn có thể sử dụng phương thức `bandNames()` của đối tượng ee.Image. Ví dụ, nếu bạn muốn kiểm tra số lượng bands trong ảnh Sentinel-1, bạn có thể làm như sau:

In [ ]:
print(f"Các bands trong ảnh Sentinel-1 GRD: {sen1col.first().bandNames().getInfo()}") # hiển thị tên các band dữ liệu trong ảnh Sentinel-1 GRD

Các bands trong ảnh Sentinel-1 GRD: ['HH', 'HV', 'angle']


### 20.4.3. Ảnh MODIS 

MODIS (Moderate Resolution Imaging Spectroradiometer) là cảm biến quang học trên các vệ tinh Terra và Aqua, cung cấp dữ liệu quan sát Trái Đất với tần suất cao và phạm vi phủ toàn cầu. Trong Google Earth Engine, MODIS được sử dụng rộng rãi để theo dõi thảm thực vật, nhiệt độ bề mặt, lớp phủ đất và các biến động môi trường thông qua các chuỗi dữ liệu dài hạn có độ phân giải từ 250 m đến 1 km.

In [ ]:
# Truy cập bộ sưu tập MODIS NDVI (Terra + Aqua) trên GEE. MODIS có 2 bộ sưu tập NDVI riêng biệt cho 2 vệ tinh Terra và Aqua, ta sẽ gộp 2 bộ sưu tập này lại với nhau để có nhiều ảnh NDVI hơn. Sau đó, ta sẽ sắp xếp theo ngày chụp ảnh tăng dần để dễ dàng truy cập ảnh mới nhất.
terra = ee.ImageCollection("MODIS/061/MOD13A2")
aqua = ee.ImageCollection("MODIS/061/MYD13A2")
modis = terra.merge(aqua).sort('system:time_start') # gộp 2 bộ sưu tập MODIS Terra và Aqua lại với nhau và sắp xếp theo ngày chụp ảnh tăng dần
print(f"Số lượng ảnh MODIS NDVI (Terra + Aqua): {modis.size().getInfo()}")

Số lượng ảnh MODIS NDVI (Terra + Aqua): 1153


- **Kiểm tra thông tin thuộc tính**

Kiểm tra thuộc tính với ảnh `MODIS` tương tự như các ảnh khác.

In [17]:
print(f"Các thuộc tính của ảnh Modis: {modis.first().propertyNames().getInfo()[:10]}") # hiển thị 10 thuộc tính đầu tiên của ảnh MODIS

Các thuộc tính của ảnh Modis: ['system:index', 'system:time_start', 'google:max_source_file_timestamp', 'num_tiles', 'system:footprint', 'system:time_end', 'system:version', 'system:id', 'system:asset_size', 'system:bands']


- **Kiểm tra bands trong ảnh**

Kiểm tra bands với ảnh `MOIDIS` cũng tương tự như với các ảnh khác.

In [18]:
print(f"Các bands trong ảnh MODIS: {modis.first().bandNames().getInfo()}") # hiển thị tên các band dữ liệu trong ảnh MODIS

Các bands trong ảnh MODIS: ['NDVI', 'EVI', 'DetailedQA', 'sur_refl_b01', 'sur_refl_b02', 'sur_refl_b03', 'sur_refl_b07', 'ViewZenith', 'SolarZenith', 'RelativeAzimuth', 'DayOfYear', 'SummaryQA']


## Tóm tắt

### Các khái niệm chính đã nắm vững:

| Đối tượng / Method | Mục đích |
|---|---|
| `ee.Image` | Ảnh raster đơn — đọc bands, thống kê vùng |
| `ee.ImageCollection` | Time-series — nền tảng để lọc và tính toán |
| `ee.Feature / FeatureCollection` | Vector data — ranh giới hành chính, điểm quan trắc |
| `.filterDate()` | Lọc theo khoảng thời gian |
| `.filterBounds()` | Lọc theo vùng địa lý |
| `.filter(ee.Filter.lt/gt/eq)` | Lọc theo metadata (cloud, orbit,...) |

### Kỹ năng bạn có thể áp dụng:
- Xác thực và kết nối GEE Python API trong môi trường Jupyter Notebook
- Truy cập và kiểm tra thông tin ảnh vệ tinh từ GEE Data Catalog (bands, properties, ngày chụp)
- Lọc chính xác `ImageCollection` theo thời gian, vùng không gian và chất lượng ảnh
- Tạo `FeatureCollection` từ dữ liệu GeoDataFrame (GeoPandas) để dùng làm AOI
- Kết hợp nhiều bộ lọc hiệu quả để chuẩn bị dữ liệu đầu vào cho các bài toán viễn thám
- Đặt nền tảng vững chắc cho các bài tiếp theo: cloud masking, tính chỉ số thực vật, tổng hợp ảnh composite